# 04 · Imágenes de satélite: el Cauca de noche
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrezconz/investigacion-aumentada-ml/blob/main/notebooks/04_imagenes_luces_cauca.ipynb)

**Pregunta:** ¿dónde se encendió el territorio entre 2014 y 2026?

**El dato:** radiancia nocturna del sensor VIIRS (satélites de la NOAA): cuánta luz emite cada píxel de unos 1 km² en la noche, en nW/cm²/sr. Se usan **los mismos meses** (enero a mayo) en ambos años para no comparar temporadas distintas.

Los mapas se recortaron al departamento del Cauca desde compuestos mensuales de Google Earth Engine.

In [ ]:
# Preparación: trae los datos del repositorio (funciona igual en Google Colab)
import os, urllib.request
REPO = "https://raw.githubusercontent.com/andrezconz/investigacion-aumentada-ml/main/datos/"

def traer(nombre):
    local = os.path.join("..", "datos", nombre)
    if os.path.exists(local):
        return local
    if not os.path.exists(nombre):
        urllib.request.urlretrieve(REPO + nombre, nombre)
    return nombre

import subprocess, sys
try:
    import rasterio
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rasterio"])
    import rasterio

In [ ]:
import numpy as np, json
luz = {}
for anio in (2014, 2026):
    with rasterio.open(traer(f"cauca_viirs_{anio}.tif")) as r:
        luz[anio] = r.read(1)
        limites = r.bounds
limite = json.load(open(traer("cauca_limite.geojson")))
print("Píxeles por año:", luz[2014].shape, "· bordes:", [round(x, 2) for x in limites])

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

ext = [limites.left, limites.right, limites.bottom, limites.top]
def dibujar_limite(ax, color="white"):
    for f in limite["features"]:
        g = f["geometry"]
        polis = g["coordinates"] if g["type"] == "MultiPolygon" else [g["coordinates"]]
        for p in polis:
            xs, ys = zip(*p[0]); ax.plot(xs, ys, color=color, lw=.8)

fig, axs = plt.subplots(1, 2, figsize=(14, 7))
for ax, anio in zip(axs, (2014, 2026)):
    ax.imshow(np.clip(luz[anio], .3, 60), extent=ext, cmap="magma", norm=LogNorm(.3, 60))
    dibujar_limite(ax); ax.set_title(f"Enero a mayo de {anio}"); ax.axis("off")
plt.show()

In [ ]:
umbral = 1.0   # nW/cm²/sr: «luz clara»
for anio in (2014, 2026):
    print(anio, "· píxeles con luz clara:", int(np.nansum(luz[anio] > umbral)))
a, b = np.nansum(luz[2014] > umbral), np.nansum(luz[2026] > umbral)
print(f"Cambio: {(b - a) / a * 100:+.0f} %")

> **Tip:** la luz es un indicador, no el desarrollo mismo. Por ejemplo, el cambio a luminarias LED puede hacer que una ciudad *parezca* apagarse, porque el sensor capta peor la luz blanca. Contrasta siempre con otra fuente.

## Del mapa al machine learning
Hasta aquí comparamos. Ahora dejemos que **K-Means** agrupe los píxeles según su luz en 2014 y en 2026 (no supervisado): nadie le dice qué es «se encendió» o «siempre oscuro».

In [ ]:
from sklearn.cluster import KMeans

valido = ~np.isnan(luz[2014]) & ~np.isnan(luz[2026])
X = np.column_stack([np.log1p(np.clip(luz[2014][valido], 0, None)),
                     np.log1p(np.clip(luz[2026][valido], 0, None))])
km = KMeans(n_clusters=4, n_init=20, random_state=0).fit(X)

centros = np.expm1(km.cluster_centers_)
orden = np.argsort(centros.sum(axis=1))
for nuevo, k in enumerate(orden, 1):
    print(f"Grupo {nuevo}: luz media 2014 = {centros[k,0]:6.2f} · 2026 = {centros[k,1]:6.2f} · píxeles = {(km.labels_ == k).sum()}")

In [ ]:
mapa = np.full(luz[2014].shape, np.nan)
reasignar = {k: i for i, k in enumerate(orden, 1)}
mapa[valido] = [reasignar[l] for l in km.labels_]

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(mapa, extent=ext, cmap="viridis")
dibujar_limite(ax, "black"); ax.axis("off")
plt.colorbar(im, ax=ax, shrink=.6, label="Grupo")
plt.title("Píxeles agrupados por su trayectoria de luz"); plt.show()

> **Pregunta para discutir:** mira la tabla de centros. ¿Qué nombre le pondrías a cada grupo? ¿En qué municipios caen los píxeles que más cambiaron?

*Fuente: NOAA VIIRS DNB (compuestos mensuales, vía Google Earth Engine). Límite departamental: Marco Geoestadístico Nacional, DANE. Elaboración propia.*